# Learned Stage-Wise Curvature Tuning on Flowers102

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Oxford Flowers-102 dataset using an ImageNet-pretrained ResNet-50.

Flowers102 is a fine-grained image-classification dataset containing many visually similar flower categories. Testing Learned SW-CT on this dataset helps evaluate whether stage-wise curvature adaptation remains useful when the target task requires distinguishing between closely related visual classes.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization improves accuracy or changes the learned stage-wise beta values.

Wall-clock execution time is recorded for each method to compare computational cost.

In [1]:
!nvidia-smi

Fri Oct  2 22:48:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 575 (delta 153), reused 181 (delta 103), pack-reused 328 (from 1)
Receiving objects: 100% (575/575), 6.93 MiB | 4.50 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
068c466 (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) Updated aircraft with complete results


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 78.5 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 57.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 74.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Flowers102 Dataset

Before running the experiments, the Flowers102 data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_flowers102",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

100%|██████████| 345M/345M [00:40<00:00, 8.62MB/s] 
100%|██████████| 502/502 [00:00<00:00, 1.61MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 16.8MB/s]

Train samples: 1020
Validation samples: 1020
Test samples: 6149


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([50, 62, 15, 96, 28,  7, 87, 73, 13, 84])


# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Flowers102 dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [3]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 23:00:37.722 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet50_seed42_epochs20.log
2026-10-02 23:00:37.889 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 188MB/s]
2026-10-02 23:00:40.378 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 23:00:40.481 | INFO     | __main__:main:116 - Number of trainable parameters: 208998
2026-10-02 23:00:40.481 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 23:00:57.279 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.742995, Accuracy: 0.00%
2026-10-02 23:00:57.554 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.778943, Val Accuracy: 18.63%
2026-10-02 23:00:57.556 | INFO     | train:linear_probe:190 - New best validation accuracy: 1

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 23:14:35.679 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet50_seed43_epochs20.log
2026-10-02 23:14:35.682 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 23:14:36.596 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 23:14:36.649 | INFO     | __main__:main:116 - Number of trainable parameters: 208998
2026-10-02 23:14:36.649 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 23:14:48.332 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.571014, Accuracy: 3.12%
2026-10-02 23:14:48.616 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.790290, Val Accuracy: 19.61%
2026-10-02 23:14:48.618 | INFO     | train:linear_probe:190 - New best validation accuracy: 19.61 at epoch 1
2026-10-02 23:14:48.691 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.675107, Accuracy: 12.50%
2026-10-02 23:14:48.996 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.487150, 

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-02 23:30:04.051 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet50_seed44_epochs20.log
2026-10-02 23:30:04.056 | INFO     | __main__:main:78 - Running on cuda
2026-10-02 23:30:05.511 | INFO     | __main__:main:105 - Testing baseline...
2026-10-02 23:30:05.589 | INFO     | __main__:main:116 - Number of trainable parameters: 208998
2026-10-02 23:30:05.590 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-02 23:30:21.473 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.725098, Accuracy: 0.00%
2026-10-02 23:30:21.874 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.741178, Val Accuracy: 19.80%
2026-10-02 23:30:21.876 | INFO     | train:linear_probe:190 - New best validation accuracy: 19.80 at epoch 1
2026-10-02 23:30:21.986 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.451101, Accuracy: 40.62%
2026-10-02 23:30:22.416 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.511231, 

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [6]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 23:46:33.313 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 23:46:33.428 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 23:46:34.630 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 23:46:34.631 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 23:46:34.750 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 23:46:34.752 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 23:46:34.752 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-02 23:46:37.087 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.731157, Accuracy: 3.12%
2026-10-02 23:47:03.415 | INFO     | train:test_epoc

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-02 23:57:05.273 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-02 23:57:05.368 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 23:57:06.429 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-02 23:57:06.430 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-02 23:57:06.547 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-02 23:57:06.548 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 23:57:06.549 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-02 23:57:08.325 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.730480, Accuracy: 0.00%
2026-10-02 23:57:36.211 | INFO     | train:test_epoc

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 00:07:36.174 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 00:07:36.270 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 00:07:37.370 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 00:07:37.371 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 00:07:37.491 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 00:07:37.492 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 00:07:37.493 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 00:07:39.355 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.703204, Accuracy: 0.00%
2026-10-03 00:08:07.205 | INFO     | train:test_epoc

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 00:18:11.425 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-10-03 00:18:11.545 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 00:18:12.795 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 00:18:12.796 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 00:18:12.899 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-03 00:18:12.900 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 00:18:12.901 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 00:18:14.919 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.748392, Accuracy: 3.12%
2026-10-03 00:18:42.901 | INFO     | tra

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 00:28:45.789 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-10-03 00:28:45.875 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 00:28:47.136 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 00:28:47.137 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 00:28:47.250 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-03 00:28:47.252 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 00:28:47.253 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 00:28:49.351 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.696371, Accuracy: 0.00%
2026-10-03 00:29:17.260 | INFO     | t

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 00:39:27.971 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-10-03 00:39:28.077 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 00:39:29.222 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 00:39:29.223 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 00:39:29.347 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-03 00:39:29.349 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 00:39:29.349 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 00:39:31.271 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.647950, Accuracy: 0.00%
2026-10-03 00:39:59.462 | INFO     | t

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [12]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 00:50:09.311 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet50_seed42_epochs30.log
2026-10-03 00:50:09.315 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 00:50:10.569 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 00:50:10.640 | INFO     | __main__:main:116 - Number of trainable parameters: 208998
2026-10-03 00:50:10.641 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 00:50:26.603 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.742995, Accuracy: 0.00%
2026-10-03 00:50:26.982 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.778943, Val Accuracy: 18.63%
2026-10-03 00:50:26.984 | INFO     | train:linear_probe:190 - New best validation accuracy: 18.63 at epoch 1
2026-10-03 00:50:27.091 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.651732, Accuracy: 21.88%
2026-10-03 00:50:27.460 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.473287, 

In [13]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 01:10:52.848 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet50_seed43_epochs30.log
2026-10-03 01:10:52.853 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 01:10:54.231 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 01:10:54.312 | INFO     | __main__:main:116 - Number of trainable parameters: 208998
2026-10-03 01:10:54.312 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 01:11:10.912 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.571014, Accuracy: 3.12%
2026-10-03 01:11:11.402 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.790290, Val Accuracy: 19.61%
2026-10-03 01:11:11.404 | INFO     | train:linear_probe:190 - New best validation accuracy: 19.61 at epoch 1
2026-10-03 01:11:11.535 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.675107, Accuracy: 12.50%
2026-10-03 01:11:11.895 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.487150, 

In [14]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 01:31:56.616 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_flowers102_resnet50_seed44_epochs30.log
2026-10-03 01:31:56.620 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 01:31:57.773 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 01:31:57.841 | INFO     | __main__:main:116 - Number of trainable parameters: 208998
2026-10-03 01:31:57.841 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 01:32:12.158 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.725098, Accuracy: 0.00%
2026-10-03 01:32:12.568 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 3.741178, Val Accuracy: 19.80%
2026-10-03 01:32:12.571 | INFO     | train:linear_probe:190 - New best validation accuracy: 19.80 at epoch 1
2026-10-03 01:32:12.694 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.451101, Accuracy: 40.62%
2026-10-03 01:32:13.102 | INFO     | train:test_epoch:90 - Epoch 2, Val Loss: 2.511231, 

In [15]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 01:52:08.606 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 01:52:08.722 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 01:52:10.015 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 01:52:10.016 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 01:52:10.123 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 01:52:10.124 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 01:52:10.125 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 01:52:12.201 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.731157, Accuracy: 3.12%
2026-10-03 01:52:38.484 | INFO     | train:test_epoc

In [16]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 02:07:28.454 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 02:07:28.583 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 02:07:29.636 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 02:07:29.637 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 02:07:29.759 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 02:07:29.760 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 02:07:29.761 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 02:07:31.802 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.730480, Accuracy: 0.00%
2026-10-03 02:07:58.298 | INFO     | train:test_epoc

In [17]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 02:22:48.455 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 02:22:48.572 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 02:22:49.674 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 02:22:49.675 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 02:22:49.785 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 02:22:49.788 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 02:22:49.788 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 02:22:51.775 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.703204, Accuracy: 0.00%
2026-10-03 02:23:19.254 | INFO     | train:test_epoc

In [18]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 02:38:15.230 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-03 02:38:15.299 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 02:38:16.220 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 02:38:16.221 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 02:38:16.315 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-03 02:38:16.316 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 02:38:16.316 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 02:38:18.224 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.748392, Accuracy: 3.12%
2026-10-03 02:38:45.264 | INFO     | tra

In [19]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 02:53:26.678 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-03 02:53:26.744 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 02:53:27.689 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 02:53:27.689 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 02:53:27.765 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-03 02:53:27.766 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 02:53:27.767 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 02:53:29.720 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.696371, Accuracy: 0.00%
2026-10-03 02:53:56.477 | INFO     | t

In [20]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds flowers102 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 03:08:25.900 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_flowers102_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-03 03:08:26.007 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 03:08:27.164 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 03:08:27.165 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 03:08:27.273 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-03 03:08:27.275 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 03:08:27.275 | INFO     | __main__:main:439 - Total trainable parameters: 209002
2026-10-03 03:08:29.287 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.647950, Accuracy: 0.00%
2026-10-03 03:08:55.988 | INFO     | t